<a href="https://colab.research.google.com/github/bwsi-hadr/04-Intro-to-networks/blob/master/04_Intro_to_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro to Networks
The study of networks investigates the relationship between discrete objects using [graphs](https://en.wikipedia.org/wiki/Graph_(discrete_mathematics))

![network_diagram](https://upload.wikimedia.org/wikipedia/commons/2/2f/Small_Network.png)

Graphs are a mathematical structure which represent a set of objects (called nodes or vertices) as well as their symmetric or asymmetric relationships (called links or edges).

They can be used to represent both physical systems (roads, pipelines, electrical grid) and abstract systems (social, collaboration, genetics).

For this course, we'll be using the python [networkx](https://networkx.org/) package, which provides many useful classes and functions for representing and doing computations with networks. 

In addition, we'll be using the [osmnx](https://osmnx.readthedocs.io/en/stable/) package, which will download openstreetmap road networks as [networkx](https://networkx.org/) objects. There are a ton of useful examples in the [examples repo](https://github.com/gboeing/osmnx-examples).

This lecture is based off of the [osmnx-examples](https://github.com/gboeing/osmnx-examples) code and the [Automating GIS Processes](https://automating-gis-processes.github.io/2018/notebooks/L6/retrieve_osm_data.html) course.

In [ ]:
import networkx as nx # need networkx >= 2.5
import osmnx as ox
import contextily as ctx
from matplotlib import pyplot as plt
from shapely.geometry import Polygon
import folium
import numpy as np
import matplotlib.patches as mpatches
import geopandas as gpd
from shapely.ops import nearest_points

## Create Our Network

In [ ]:
# Specify the name that is used to search for the data
place_name = "Cambridge, MA, USA"

# Fetch OSM street network from the location
# this will take a while because there's a lot of streets
graph = ox.graph_from_place(place_name)

In [ ]:
# metadata about the graph object
graph.graph

Here are the explanations:

- **created_date**:
   The timestamp indicating when the graph was created.
- **created_with**: `'OSMnx 1.9.3'`
   The version of OSMnx used to generate the graph.
- **crs**: `'epsg:4326'`
   The coordinate reference system of the graph, in this case EPSG:4326 (WGS84 geographic coordinates).
- **simplified**: `True`
   Indicates that the graph has been simplified using OSMnx’s simplification process, which merges unnecessary intermediate nodes into consolidated edges for cleaner analysis.

## Saving and Loading

We can save the graph to a file with `osmnx.io.save_graphml` so that we don't have to wait for it to download again next time.

In [ ]:
ox.io.save_graphml(graph, 'cambridge_osmnx.graphml')

To load graph from local file, use `osmnx.io.load_graphml`

In [ ]:
graph = ox.io.load_graphml('cambridge_osmnx.graphml')

## Plot Our Graph

Let's take a look at the graph

In [ ]:
fig, ax = ox.plot_graph(graph)

## Projection

By default, osmnx graphs are created in WGS84 (epsg:4326)

But we can project the graph into epsg:3857 so that units are in meters by using the `osmnx.project_graph()` function.

In [ ]:
graph_proj = ox.project_graph(graph, to_crs='epsg:3857')

We can then convert the projected graph to a geodataframe with `osmnx.graph_to_gdfs()`

In [ ]:
graph_nodes_gdf, graph_edges_gdf = ox.graph_to_gdfs(graph_proj)

# Check the CRS of the nodes GeoDataFrame
graph_nodes_gdf.crs

See the graph nodes

In [ ]:
graph_nodes_gdf

See the edges

In [ ]:
graph_edges_gdf

In [ ]:
# we can also get a gdf for the footprint of the place
place_footprint = ox.geocode_to_gdf(place_name)
graph_area = place_footprint.to_crs('epsg:3857')
graph_area

In [ ]:
graph_area.plot()

## Exercise 1: Simple Network Calculations

### Exercise 1.1: Total Area

What is the total area of cambridge in square meters?

In [ ]:
def findTotalArea(gdf):
    """
    input: gdf of geometries
    output: total area of all geometries in the gdf
    """

    # your code here
    
    pass

Test your function

In [ ]:
findTotalArea(graph_area)
# should return 33732821.38796457 (or something close)

### Exercise 1.2: Street Segments

How many street segments in Cambridge are longer than 100 meters?

In [ ]:
def roadSegmentCount(gdf, min_length=100):
    """
    input: gdf of roads, min_length in meters
    output: number of road segments longer than min_length
    """

    # your code here
    
    pass

Test your function

In [ ]:
roadSegmentCount(graph_edges_gdf, min_length=100)
# should return 3407

## Road network statistics

In [ ]:
# we can get basic stats about the network:
ox.stats.basic_stats(graph_proj)

Circuity is the ratio of network distance (distance traveled along roads) to euclidean distance (straight line). Higher circuity means greater inefficiency in traveling along the roads.

Check the [documentation](https://osmnx.readthedocs.io/en/stable/osmnx.html#module-osmnx.stats) for more info about each statistic

In [ ]:
# if you pass in the area, you can get density info
area_cambridge_sqm = graph_area['geometry'].area
ox.basic_stats(graph_proj, area=area_cambridge_sqm)

## Getting data other ways
In the above example, we used the name of a place to get the data that we wanted.

However, there are other ways to specify locations which may be more convenient.

For example, you can get graphs from [address](https://osmnx.readthedocs.io/en/stable/osmnx.html?highlight=poly#osmnx.core.graph_from_address), [bounding box](https://osmnx.readthedocs.io/en/stable/osmnx.html?highlight=poly#osmnx.core.graph_from_bbox), [points](https://osmnx.readthedocs.io/en/stable/osmnx.html?highlight=poly#osmnx.core.graph_from_bbox), or [shapely polygon object](https://osmnx.readthedocs.io/en/stable/osmnx.html?highlight=poly#osmnx.core.graph_from_polygon)

The types of networks you can get are:

-    `drive` - get drivable public streets (but not service roads)
-    `drive_service` - get drivable streets, including service roads
-    `walk` - get all streets and paths that pedestrians can use (this network type ignores one-way directionality)
-    `bike` - get all streets and paths that cyclists can use
-    `all` - download all non-private OSM streets and paths
-    `all_private` - download all OSM streets and paths, including private-access ones

Below is an example of getting the pedestrains network (`walk` type) within a 1 mile distance from MIT main campus:

In [ ]:
# Coordinates of the MIT main campus in Decimal Degrees
coordinates = [(-71.092562, 42.357602), (-71.080155, 42.361553),
               (-71.089817, 42.362584), (-71.094688, 42.360198)]

# Create a Shapely polygon from the coordinate-tuple list
poly = Polygon(coordinates)

# convert to meters from latlong by projecting to epsg:3857
poly_m, poly_crs_m = ox.projection.project_geometry(poly, to_crs='epsg:3857')

# put a buffer of 1 mile (1609.34 m) to get everything w/in 1 mile
poly_with_buffer_m = poly_m.buffer(1609.34)

# convert back to lat-long
poly_with_buffer_ll, poly_crs_ll = ox.projection.project_geometry(poly_with_buffer_m, 
                                                        crs=poly_crs_m, 
                                                        to_crs='epsg:4326', 
                                                        to_latlong=True)

'''
# get all walkable roads within poly
walkable_roads_near_mit = ox.graph_from_polygon(poly_with_buffer_ll, 
                                       network_type='walk',
                                       clean_periphery=True)
'''
# Get all walkable roads within poly
walkable_roads_near_mit = ox.graph_from_polygon(
    poly_with_buffer_ll, 
    network_type='walk'
)


In [ ]:
# let's plot it
fig, ax = ox.plot_graph(walkable_roads_near_mit, figsize=(8,10), node_size=10)

### Edges

In [ ]:
# example of a road
all_edges_list = list(walkable_roads_near_mit.edges) # put the id for each edge in a list
print(all_edges_list[0]) # id of first edge
print(walkable_roads_near_mit.edges[all_edges_list[0]]) # get details for first edge

In [ ]:
# get attribute for all edges
walkable_roads_near_mit.edges(data='name')

### Nodes

In [ ]:
# example of a node (intersection)
all_nodes_list = list(walkable_roads_near_mit.nodes)
print(all_nodes_list[0]) # id of first node
# details for first node
print(walkable_roads_near_mit.nodes[all_nodes_list[0]])

In [ ]:
# get attribute for all nodes
walkable_roads_near_mit.nodes(data='street_count')

## Exercise 2: Walkable Roads

How many walkable roads near MIT are longer than 100 meters?

In [ ]:
def walkableRoadsCount(nw, min_length=100):
    """
    input:
        nw: networkx graph of roads
        min_length: minimum length of road segment in meters
    output:
        number of walkable road segments longer than min_length
    """

    # your code here

    pass

Test your function

In [ ]:
walkableRoadsCount(walkable_roads_near_mit, min_length=100)
# should return 1328

## Routing
We can get the node closest to a point using the `ox.distance.nearest_node` function.

In [ ]:
orig_node = ox.distance.nearest_nodes(walkable_roads_near_mit, -71.092562, 42.357602)

# let's calculate the route to a random destination node
rand_node = np.random.choice(walkable_roads_near_mit.nodes)
print(walkable_roads_near_mit[rand_node])

route = ox.routing.shortest_path(walkable_roads_near_mit, 
                         orig_node, rand_node, 
                         weight='length')

fig, ax = ox.plot_graph_route(walkable_roads_near_mit, route, node_size=0)
shortest_path_length = nx.shortest_path_length(walkable_roads_near_mit, orig_node, 
                              rand_node, weight='length')
print(f'length of shortest walkable path from node {orig_node} to {rand_node} is {shortest_path_length:.2f}m') # the {:.2f} syntax means round off to 2 digits after decimal pt

The `shortest_path` function in networkx uses [Dijkstra's algorithm](https://en.wikipedia.org/wiki/Dijkstra%27s_algorithm) to efficiently calculate the shortest path between two nodes.

The networkx library also provides many other useful functions on graphs; for instance, the [Floyd-Warshall algorithm](https://en.wikipedia.org/wiki/Floyd%E2%80%93Warshall_algorithm), which efficiently computes the shortest paths between all pairs of nodes, and the [A* (A-star) algorithm](https://en.wikipedia.org/wiki/A*_search_algorithm), which is similar to Dijkstra's algorithm, except it runs faster by taking advantage of a distance "approximation" that can be calculated between two nodes without knowing the shortest path outright.

Check out these links if you want to learn more about these algorithms, and check out the [networkx documentation](https://networkx.github.io/documentation/networkx-2.4/index.html) if you want to learn more about all the methods networkx provides.

## Building Outlines (Footprint)
We can get the footprint of objects such as buildings as geodataframes with `osmnx`. Here we'll get the building footprints near MIT.

In [ ]:
# we can get the footprints of objects within this poly
building_footprints = ox.features.features_from_polygon(poly_with_buffer_ll, tags={'building':True})

You can perform any query using OSM tags: https://wiki.openstreetmap.org/wiki/Map_features

Note that if you include multiple tags, it gets the **union** of the results, not the intersection

In [ ]:
road_footprints = ox.features.features_from_polygon(poly_with_buffer_ll, tags={'highway':True}) # get roads with geometry
# note that most roads do not have footprint information

In [ ]:
# project into UTM for info in meters
building_footprints_proj = ox.projection.project_gdf(building_footprints)
road_footprints_proj = ox.projection.project_gdf(road_footprints)
building_footprints_proj

In [ ]:
# column names
building_footprints_proj.columns.values

In [ ]:
# drop rows where there's no shape info
building_footprints_proj = building_footprints_proj.dropna(subset=['geometry'])
road_footprints = road_footprints.dropna(subset=['geometry'])

In [ ]:
fig, ax = ox.plot_footprints(building_footprints_proj)

In [ ]:
# plot roads with footprints
fig, ax = ox.plot_footprints(road_footprints_proj) # note that this is only a small subset of all roads

In [ ]:
# save the geodataframe to a file to persist
# can choose to write only a subset of columns if you want
building_footprints_proj[['description', 'operator', 'railway', 'geometry', 'attribution',
       'source', 'addr:city', 'addr:housenumber', 'addr:postcode',
       'addr:state', 'addr:street', 'amenity']].to_file('buildings_around_mit.geojson', driver='GeoJSON')


## Exercise 3: Plot the Walkable Roads and Buildings

Plot the walkable roads and buildings in the area near MIT in the same figure.

*Note: The `osmnx` plotting features use matplotlib on the backend. Check [the documentation](https://osmnx.readthedocs.io/en/stable/osmnx.html#osmnx.plot.plot_graph) for more formatting options for plotting*

In [ ]:
from matplotlib.lines import Line2D

# create figure
fig, ax = plt.subplots(figsize=(12,12))

# add road footprints
# your code here

# add building footprints
# your code here

# Challenge 1: add legend
# add legend for building footprints, patch type
# your code here

# add legend for road footprints, line type
# your code here

# add legend to map
# your code here

# Challenge 2: add basemap
# your code here
# hint: use ctx.add_basemap()

# add figure title
# your code here

# show the plot
plt.show()

Now, go back to one of the previous exercises and plot all the nodes on the periphery of Cambridge and the roads in Cambridge in the same figure.

In [ ]:
# create figure
fig, ax = plt.subplots(figsize=(12,12))

# Define the periphery as the convex hull of the nodes using union_all()
# your code here

# convert to Geodataframe
# your code here

# keep only the boundary of the periphery
# hint: use the `boundary` property of the GeoDataFrame

# plot the periphery
# your code here

# plot the roads
# your code here

# plot the nodes
# your code here

# Add legends
# hint: use `Line2D`

plt.title('Nodes on the Periphery and Roads in Cambridge')
plt.show()

## Interactive graph with folium

[Folium](https://python-visualization.github.io/folium/latest/) is a library which creates interactive web maps. This can be used to visualize and explore the data. However, it is slower to load and cannot scale to as large networks as the standard matplotlib plotting

In [ ]:
# Convert the walkable roads graph to GeoDataFrames
graph_nodes_gdf, graph_edges_gdf = ox.graph_to_gdfs(walkable_roads_near_mit)

# Use the explore method to create an interactive web map
map_edges = graph_edges_gdf.explore()

# Optionally, you can save the map to an HTML file to view it in a browser
map_edges.save('walkable_roads_near_mit.html')

# To display the map in a Jupyter notebook
map_edges


In [ ]:
import geopandas as gpd
from shapely.geometry import LineString

# Extract the route geometry
route_nodes_gdf = graph_nodes_gdf.loc[route]
route_line = LineString(route_nodes_gdf.geometry.tolist())

# Create a GeoDataFrame for the route
route_gdf = gpd.GeoDataFrame(geometry=[route_line], crs=graph_edges_gdf.crs)

# Add the route to the map
map_edges = route_gdf.explore(m=map_edges, color='#ff0000', tooltip=False)

# To display the map in a Jupyter notebook
map_edges

## Get places of interest
OSM can also give places of interest, like restaurants, pharmacies, hospitals, and toilets. The full list is available [here](https://wiki.openstreetmap.org/wiki/Key:amenity).

Let's look at the POIs near MIT.

In [ ]:
fast_food = ox.features.features_from_polygon(poly_with_buffer_ll, tags={'amenity': 'fast_food'})
fast_food

In [ ]:
walkable_nodes, walkable_edges = ox.graph_to_gdfs(walkable_roads_near_mit)

In [ ]:
# plot roads, buildings, and fast_food
fig, ax = plt.subplots(1,1, figsize=[20,20])
building_footprints.plot(ax=ax)
walkable_edges.plot(ax=ax, color='black')
fast_food.plot(ax=ax, color='#FF0099', markersize=50)
plt.show()

## Exercise 4: Group Challenge

In groups, pick a location, ask an interesting question, do an analysis to answer that question.

If you want to explore other options other than OSM networks, here are some other databases for inspirations:

- [Kaggle](https://www.kaggle.com/) — large repository of datasets across domains  
- [NYC Open Data](https://opendata.cityofnewyork.us/) — extensive datasets on NYC  
- [MTA Subway Hourly Ridership](https://data.ny.gov/Transportation/MTA-Subway-Hourly-Ridership-2020-2024/wujg-7c2s/about_data) — subway ridership data  
- [ArcGIS Living Atlas](https://livingatlas.arcgis.com/en/browse/#d=2) — curated global GIS layers  
- [OpenStreetMap (Geofabrik)](https://download.geofabrik.de/) — worldwide vector map data  
- [US Census Bureau TIGER/Line Shapefiles](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html) — US geospatial data  
- [Natural Earth](https://www.naturalearthdata.com/) — global cultural and physical data layers  
- [Overpass Turbo](https://overpass-turbo.eu/) — extract targeted data from OpenStreetMap  
- [USGS Earth Explorer](https://earthexplorer.usgs.gov/) — remote sensing imagery  
- [European Union Open Data Portal](https://data.europa.eu/en) — EU-wide data portal  
- [Chicago Data Portal](https://data.cityofchicago.org/) — data from the city of Chicago  
- [Data.gov](https://www.data.gov/) — US government data portal  
- [World Bank Open Data](https://data.worldbank.org/) — global economic and social data  


In [ ]:
# Try starting by carefully describing your question, and then find a way to answer that!